# 📈 Notebook 04: SHAP Explainability & Attribution Analysis
This notebook demonstrates how to compute and visualize SHAP values for the trained XGBoost model, explaining both global feature importances and local predictions for individual customers.

In [ ]:
import sys
import os
import pandas as pd
import joblib
import matplotlib.pyplot as plt

# Add source path
sys.path.append('../')
from src.predict import load_artifacts
from src.explain import get_shap_explainer, global_feature_importance_plot, local_explanation_plot, get_top_factors

### 1. Load Artifacts and Scale Data
Load test set, model, scaler, and features.

In [ ]:
df_clean = pd.read_pickle('../data/processed/cleaned_data.pkl')
X = df_clean.drop(columns=['Churn'], errors='ignore')
y = df_clean['Churn']

from sklearn.model_selection import train_test_split
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model, scaler, features = load_artifacts()
X_test_scaled = pd.DataFrame(scaler.transform(X_test), index=X_test.index, columns=X_test.columns)
print("Preprocessed Test Set scaled and loaded.")

### 2. Instantiate SHAP Explainer
Create the SHAP TreeExplainer using our trained XGBoost model.

In [ ]:
explainer = get_shap_explainer(model, X_test_scaled)

### 3. Global Interpretability
Plot a beeswarm chart representing the global importances and directional impacts of each feature across the test dataset.

In [ ]:
# Plot global SHAP summary
fig_global = global_feature_importance_plot(explainer, X_test_scaled.sample(300, random_state=42))
plt.show()

### 4. Local Explanations
Choose a specific customer from the test set and explain the individual prediction using a SHAP waterfall plot.

In [ ]:
sample_customer_id = X_test_scaled.index[0]
customer_row = X_test_scaled.loc[[sample_customer_id]]
print(f"Explaining prediction for customer: {sample_customer_id}")

# Get top numerical factors
top_factors = get_top_factors(explainer, customer_row, top_n=5)
for feature, impact in top_factors:
    print(f"- {feature}: {impact:+.4f}")

# Plot waterfall
fig_local = local_explanation_plot(explainer, customer_row)
plt.show()